In [1]:
import torch
import warnings
import os
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, GenerationConfig
from transformers import logging
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

logging.set_verbosity_error()
warnings.filterwarnings("ignore")

### 1. Загрузка модели и токенайзера

In [ ]:
MODEL_NAME = "codellama/CodeLlama-13b-Instruct-hf"

QUNTIZATION_CONFIG = BitsAndBytesConfig(
    load_in_4bit=True, 
    bnb_4bit_compute_dtype=torch.bfloat16,       
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

MODEL = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    quantization_config=QUNTIZATION_CONFIG,
    device_map='auto')

TOKENIZER = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
TOKENIZER.use_default_system_prompt = False

### 2. Выставление конфига

In [ ]:
GENERATION_CONFIG = GenerationConfig(
    max_new_tokens=512,
    do_sample=False,
    num_beams=1,
    num_beam_groups=1,
    diversity_penalty=1.0,
    num_return_sequences=1,
    eos_token_id=TOKENIZER.eos_token_id,
)

### 3. Функции для генерации промта, запуска модели и сохранения результатов

In [ ]:
def generate_prompt(row):
    lang = "C++"
    if row["repository"] in ["openssl", "redis"]:
        lang = "C"
    func_name = row["fname"].replace("\n", " ")
    func_name = " ".join(func_name.split())
    function_description = row["doc"]
    system_message = {
        "role": "system",
        # "content": f"You're a specialized AI assisting with generating function code on {lang}. You are very good at generating code.\n",
        "content": f"\nGenerate a function on {lang} programming language.\n",
    }

    prompt = {
        "role": "user",
        "content": (
            f'Function name `{func_name}`.\nFunction description: "{function_description}".'
        ),
    }
    return [system_message, prompt]

def inference(sample):
    prompt = generate_prompt(sample)
    inputs = TOKENIZER.apply_chat_template(
        prompt, add_generation_prompt=True, return_tensors="pt"
    ).to(MODEL.device)
    outputs = MODEL.generate(
        inputs,
        generation_config=GENERATION_CONFIG,
    )
    return [TOKENIZER.decode(
        output[len(inputs[0]) :], skip_special_tokens=True
    ).strip() for output in outputs]


def save_predictions(code_, save_path):
    # 
    k = 1
    df_to_save = pd.DataFrame.from_dict(code_, orient='index', columns=[f'{i}' for i in range(k)])
    df_to_save.to_csv(save_path)

### 4. Загрузка датасета и запуск модели

In [18]:
df = pd.read_json('bench-v0.6.json')
df["79D9B4DB619F85EB"].iloc[0]

[{'id': '2D57CE778BDD1AD4',
  'name': 'btVector3::operator+=',
  'path': 'bullet3/src/LinearMath/btVector3.h',
  'start': {'line': 159, 'col': 2},
  'end': {'line': 171, 'col': 2},
  'code': '\t{\n\t\tm_floats[0] += v.m_floats[0];\n\t\tm_floats[1] += v.m_floats[1];\n\t\tm_floats[2] += v.m_floats[2];\n\t\treturn *this;\n\t}\n\n\t/**@brief Subtract a vector from this one\n   * @param The vector to subtract */\n\tSIMD_FORCE_INLINE btVector3& operator-=(const btVector3& v)\n\t{\n\t\tm_floats[0] -= v.m_floats[0];\n\t\tm_floats[1] -= v.m_floats[1];\n\t\tm_floats[2] -= v.m_floats[2];\n\t\treturn *this;\n\t}\n\n\t/**@brief Scale the vector\n   * @param s Scale factor */\n\tSIMD_FORCE_INLINE btVector3& operator*=(const btScalar& s)\n\t{\n\t\tm_floats[0] *= s;\n\t\tm_floats[1] *= s;\n\t\tm_floats[2] *= s;\n\t\treturn *this;\n\t}\n\n\t/**@brief Inversely scale the vector \n   * @param s Scale factor to divide by */\n\tSIMD_FORCE_INLINE btVector3& operator/=(const btScalar& s)\n\t{\n\t\tbtFullAsse

In [ ]:
code = {}

for ind, row in df.iterrows():
    print(f"{ind}/{len(df.shape[0])} in progress.")
    code[row['Unnamed: 0']] = inference(row)
    
save_predictions(code, "generations.csv")
